In [ ]:
import os
import time
import urllib.request
import zipfile
import pickle  # Added for saving vocabulary
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from pycocotools.coco import COCO
import nltk
from collections import Counter

# Download necessary NLTK data
nltk.download('punkt')
nltk.download('punkt_tab')

# Create directories
os.makedirs('data/val2017', exist_ok=True)
os.makedirs('data/annotations', exist_ok=True)

# Download val images (~1GB)
val_url = 'http://images.cocodataset.org/zips/val2017.zip'
val_output = 'data/val2017.zip'
print("Downloading validation images...")
urllib.request.urlretrieve(val_url, val_output)
print("Unzipping validation images...")
with zipfile.ZipFile(val_output, 'r') as zip_ref:
    zip_ref.extractall('data/')

# Download annotations (~241MB) with retry
anno_url = 'http://images.cocodataset.org/annotations/annotations_trainval2017.zip'
anno_output = 'data/annotations_trainval2017.zip'
retry_attempts = 5
download_success = False
for attempt in range(retry_attempts):
    print(f"Attempt {attempt+1}/{retry_attempts} to download annotations...")
    if os.path.exists(anno_output):
        os.remove(anno_output)  # Remove incomplete file
    try:
        urllib.request.urlretrieve(anno_url, anno_output)
        download_success = True
        print("Download successful.")
        break
    except Exception as e:
        print(f"Download failed: {e}. Retrying in 10 seconds...")
        time.sleep(10)

if not download_success:
    print("Failed to download annotations after multiple retries.")
else:
    # Unzip annotations
    print("Unzipping annotations...")
    try:
        with zipfile.ZipFile(anno_output, 'r') as zip_ref:
            zip_ref.extractall('data/annotations/')
        print("Annotations unzipped successfully.")
    except Exception as e:
        print(f"Failed to unzip annotations: {e}")

# Vocabulary class
class Vocabulary:
    def __init__(self, freq_threshold):
        self.itos = {0: "<pad>", 1: "<start>", 2: "<end>", 3: "<unk>"}
        self.stoi = {"<pad>": 0, "<start>": 1, "<end>": 2, "<unk>": 3}
        self.freq_threshold = freq_threshold

    def __len__(self):
        return len(self.itos)

    @staticmethod
    def tokenizer_english(text):
        return [tok.lower() for tok in nltk.word_tokenize(text)]

    def build_vocabulary(self, sentence_list):
        counts = Counter()
        for sentence in sentence_list:
            counts.update(self.tokenizer_english(sentence))
        for word, freq in counts.items():
            if freq > self.freq_threshold:
                self.stoi[word] = len(self.itos)
                self.itos[len(self.itos)] = word

    def numericalize(self, text):
        tokenized_text = self.tokenizer_english(text)
        return [
            self.stoi.get(token, self.stoi["<unk>"]) for token in tokenized_text
        ]


# Dataset class
class CocoDataset(Dataset):
    def __init__(self, root_dir, anno_file, vocab=None, transform=None, max_len=50, limit=None):
        self.root_dir = root_dir
        self.coco = COCO(anno_file)
        self.ids = list(self.coco.anns.keys())
        if limit is not None:
            self.ids = self.ids[:limit]

        self.transform = transform or transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
        self.max_len = max_len
        if vocab is None:
            captions = [self.coco.anns[ann_id]['caption'] for ann_id in self.ids]
            self.vocab = Vocabulary(freq_threshold=5)
            self.vocab.build_vocabulary(captions)
        else:
            self.vocab = vocab
        self.vocab_size = len(self.vocab)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        ann_id = self.ids[idx]
        caption = self.coco.anns[ann_id]['caption']
        img_id = self.coco.anns[ann_id]['image_id']
        img_info = self.coco.loadImgs(img_id)[0]
        path = os.path.join(self.root_dir, img_info['file_name'])

        image = Image.open(path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        caption_numerical = [self.vocab.stoi["<start>"]]
        caption_numerical += self.vocab.numericalize(caption)
        caption_numerical.append(self.vocab.stoi["<end>"])

        if len(caption_numerical) < self.max_len:
            caption_numerical += [self.vocab.stoi["<pad>"]] * (self.max_len - len(caption_numerical))
        else:
            caption_numerical = caption_numerical[:self.max_len]

        return image, torch.tensor(caption_numerical, dtype=torch.long)

def collate_fn(batch):
    images, captions = zip(*batch)
    images = torch.stack(images)
    captions = torch.stack(captions)
    return images, captions

# Model definitions
class EncoderCNN(nn.Module):
    def __init__(self, embed_size=256):
        super(EncoderCNN, self).__init__()
        resnet = models.resnet50(pretrained=True)
        for param in resnet.parameters():
            param.requires_grad = False
        modules = list(resnet.children())[:-1]
        self.resnet = nn.Sequential(*modules)
        self.linear = nn.Linear(resnet.fc.in_features, embed_size)
        self.bn = nn.BatchNorm1d(embed_size)

    def forward(self, images):
        features = self.resnet(images)
        features = features.view(features.size(0), -1)
        features = self.bn(self.linear(features))
        return features

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-torch.log(torch.tensor(10000.0)) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:x.size(0), :]

class DecoderTransformer(nn.Module):
    def __init__(self, embed_size, hidden_size, vocab_size, num_layers=3, num_heads=8):
        super(DecoderTransformer, self).__init__()
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.pos_encode = PositionalEncoding(embed_size)
        decoder_layer = nn.TransformerDecoderLayer(d_model=embed_size, nhead=num_heads)
        self.transformer = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.fc_out = nn.Linear(embed_size, vocab_size)

    def forward(self, features, captions, tgt_mask=None):
        memory = features.unsqueeze(1).transpose(0, 1)
        embed = self.embed(captions)
        embed = self.pos_encode(embed.transpose(0, 1))
        if tgt_mask is None:
            tgt_mask = nn.Transformer.generate_square_subsequent_mask(embed.size(0)).to(embed.device)
        output = self.transformer(embed, memory, tgt_mask=tgt_mask)
        output = self.fc_out(output.transpose(0, 1))
        return output

class ImageCaptioner(nn.Module):
    def __init__(self, embed_size, hidden_size, vocab_size, num_layers=3, num_heads=8):
        super(ImageCaptioner, self).__init__()
        self.encoder = EncoderCNN(embed_size)
        self.decoder = DecoderTransformer(embed_size, hidden_size, vocab_size, num_layers, num_heads)

    def forward(self, images, captions):
        features = self.encoder(images)
        outputs = self.decoder(features, captions[:, :-1])
        return outputs

# Training
def train():
    embed_size = 256
    hidden_size = 512
    num_layers = 3
    num_heads = 8
    lr = 3e-4
    epochs = 10
    batch_size = 32

    anno_file = 'data/annotations/annotations/captions_val2017.json'
    dataset = CocoDataset(root_dir='data/val2017', anno_file=anno_file)
    vocab_size = dataset.vocab_size

    # Save vocabulary
    with open('vocab.pkl', 'wb') as f:
        pickle.dump(dataset.vocab, f)

    train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = ImageCaptioner(embed_size, hidden_size, vocab_size, num_layers, num_heads).to(device)
    criterion = nn.CrossEntropyLoss(ignore_index=dataset.vocab.stoi['<pad>'])
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}', leave=True)
        for images, captions in progress_bar:
            images, captions = images.to(device), captions.to(device)
            outputs = model(images, captions)
            loss = criterion(outputs.view(-1, vocab_size), captions[:, 1:].contiguous().view(-1))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            progress_bar.set_postfix({'batch_loss': f'{loss.item():.4f}'})
        avg_loss = total_loss / len(train_loader)
        print(f'Epoch {epoch+1}/{epochs}, Average Loss: {avg_loss:.4f}')

        torch.save(model.state_dict(), f'model_epoch_{epoch+1}.pth')

    print('Training complete!')

# Evaluation
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

class CocoEvalDataset(Dataset):
    def __init__(self, root_dir, anno_file, transform=None):
        self.root_dir = root_dir
        self.coco = COCO(anno_file)
        self.img_ids = list(self.coco.imgs.keys())
        self.transform = transform or transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        img_info = self.coco.imgs[img_id]
        path = os.path.join(self.root_dir, img_info['file_name'])
        image = Image.open(path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)
        captions = [ann['caption'] for ann in anns]
        return image, captions, img_id

def collate_fn_eval(batch):
    images, captions_lists, img_ids = zip(*batch)
    images = torch.stack(images)
    return images, captions_lists, img_ids

def greedy_decode(decoder, features, vocab, max_len=50):
    batch_size = features.size(0)
    device = features.device
    captions = torch.ones(batch_size, 1).fill_(vocab.stoi["<start>"]).long().to(device)
    for _ in range(max_len):
        outputs = decoder(features, captions)
        next_word = outputs[:, -1, :].argmax(dim=1).unsqueeze(1)
        captions = torch.cat([captions, next_word], dim=1)
        if all(captions[:, -1] == vocab.stoi["<end>"]):
            break
    decoded = []
    for cap in captions:
        cap_list = [vocab.itos[idx.item()] for idx in cap[1:]]
        if '<end>' in cap_list:
            cap_list = cap_list[:cap_list.index('<end>')]
        decoded.append(cap_list)
    return decoded

def compute_bleu(preds, targets):
    smooth = SmoothingFunction().method1
    return corpus_bleu(targets, preds, smoothing_function=smooth)

def evaluate(model_path='model_epoch_10.pth'):
    embed_size = 256
    hidden_size = 512
    num_layers = 3
    num_heads = 8

    anno_file = 'data/annotations/annotations/captions_val2017.json'
    train_dataset = CocoDataset(root_dir='data/val2017', anno_file=anno_file)
    vocab = train_dataset.vocab
    vocab_size = train_dataset.vocab_size

    val_dataset = CocoEvalDataset(root_dir='data/val2017', anno_file=anno_file)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn_eval)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = ImageCaptioner(embed_size, hidden_size, vocab_size, num_layers, num_heads).to(device)
    model.load_state_dict(torch.load(model_path))
    model.eval()

    preds = []
    targets = []
    with torch.no_grad():
        for images, captions_lists, _ in val_loader:
            images = images.to(device)
            features = model.encoder(images)
            batch_preds = greedy_decode(model.decoder, features, vocab, max_len=50)
            preds.extend(batch_preds)
            batch_targets = []
            for caps in captions_lists:
                tokenized_caps = [Vocabulary.tokenizer_english(cap) for cap in caps]
                batch_targets.append(tokenized_caps)
            targets.extend(batch_targets)

    bleu = compute_bleu(preds, targets)
    print(f'BLEU Score: {bleu:.4f}')

# Inference
def beam_search_decode(decoder, features, vocab, beam_width=3, max_len=50):
    device = features.device
    start = [vocab.stoi["<start>"]]
    start_word = [(start, 0.0)]
    while len(start_word[0][0]) < max_len:
        temp = []
        for s in start_word:
            captions = torch.tensor([s[0]], dtype=torch.long).to(device)
            outputs = decoder(features, captions)
            probs = torch.log_softmax(outputs[:, -1, :], dim=1)
            top_probs, top_words = probs.topk(beam_width, dim=1)
            for i in range(beam_width):
                next_seq = s[0] + [top_words[0, i].item()]
                next_score = s[1] + top_probs[0, i].item()
                temp.append((next_seq, next_score))
        start_word = sorted(temp, reverse=True, key=lambda x: x[1])[:beam_width]
        if start_word[0][0][-1] == vocab.stoi["<end>"]:
            break
    seq = start_word[0][0]
    cap_list = [vocab.itos[idx] for idx in seq[1:-1]] if seq[-1] == vocab.stoi["<end>"] else [vocab.itos[idx] for idx in seq[1:]]
    return cap_list

def generate_caption(image_path, model_path='model_epoch_10.pth', vocab_path='vocab.pkl'):
    embed_size = 256
    hidden_size = 512
    num_layers = 3
    num_heads = 8

    # Load vocabulary
    with open(vocab_path, 'rb') as f:
        vocab = pickle.load(f)
    vocab_size = len(vocab)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = ImageCaptioner(embed_size, hidden_size, vocab_size, num_layers, num_heads).to(device)
    model.load_state_dict(torch.load(model_path))
    model.eval()



    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    image = Image.open(image_path).convert('RGB')
    image = transform(image).unsqueeze(0).to(device)

    features = model.encoder(image)
    caption = beam_search_decode(model.decoder, features, vocab)
    return ' '.join(caption)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Unzipping validation images...
Attempt 1/5 to download annotations...
Download successful.
Unzipping annotations...
Annotations unzipped successfully.
